# Airline Arrival Delay Prediction

This notebook focuses only on **feature selection and prediction** using the cleaned airline dataset.

Goal: Predict whether a flight will arrive late using schedule-based features.

## 1. Start Spark Session

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("AirlineArrivalDelayPrediction") \
    .config("spark.driver.memory", "6g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

print("Spark session started.")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/28 13:59:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark session started.


26/04/28 13:59:43 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


## 2. Load Cleaned Dataset

The cleaned data is stored in the Spark output folder `cleaned_airline_data`.

In [2]:
df = spark.read.csv(
    "cleaned_airline_data",
    header=True,
    inferSchema=True
)

print("Rows:", df.count())
print("Columns:", len(df.columns))
df.printSchema()

[Stage 2:>                                                        (0 + 10) / 11]

Rows: 7070784
Columns: 42
root
 |-- Year: integer (nullable = true)
 |-- Quarter: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayofMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- FlightDate: date (nullable = true)
 |-- Reporting_Airline: string (nullable = true)
 |-- DOT_ID_Reporting_Airline: integer (nullable = true)
 |-- IATA_CODE_Reporting_Airline: string (nullable = true)
 |-- Origin: string (nullable = true)
 |-- OriginCityName: string (nullable = true)
 |-- OriginState: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- DestCityName: string (nullable = true)
 |-- DestState: string (nullable = true)
 |-- CRSDepTime: integer (nullable = true)
 |-- DepTime: integer (nullable = true)
 |-- DepDelay: double (nullable = true)
 |-- DepDelayMinutes: double (nullable = true)
 |-- CRSArrTime: integer (nullable = true)
 |-- ArrTime: integer (nullable = true)
 |-- ArrDelay: double (nullable = true)
 |-- ArrDelayMinutes: d

In [50]:
from pyspark.sql.functions import col, when, lag, coalesce
from pyspark.sql.window import Window

# 1. Remove noisy rows
df = df.filter(col("Cancelled") == 0)
df = df.filter(col("Diverted") == 0)

# 2. Time bucket feature
df = df.withColumn(
    "time_bucket",
    when(col("dep_hour") < 6, "early_morning")
    .when(col("dep_hour") < 12, "morning")
    .when(col("dep_hour") < 18, "afternoon")
    .otherwise("evening")
)

# 3. Distance bucket
df = df.withColumn(
    "distance_bucket",
    when(col("Distance") < 500, "short")
    .when(col("Distance") < 1500, "medium")
    .otherwise("long")
)

# 4. Previous flight delay (POWERFUL)
window_spec = Window.partitionBy("Reporting_Airline", "Origin").orderBy("CRSDepTime")

df = df.withColumn(
    "prev_dep_delay",
    lag("DepDelay").over(window_spec)
)

df = df.withColumn(
    "prev_dep_delay",
    coalesce(col("prev_dep_delay"), col("DepDelay"))
)

## 3. Feature Engineering

We add two simple schedule-based features:

- `is_peak_hour`: whether the flight departs during morning/evening peak hours
- `is_weekend`: whether the flight is scheduled on a weekend

In [51]:
from pyspark.sql.functions import col, when

df = df.withColumn(
    "is_peak_hour",
    when((col("dep_hour") >= 6) & (col("dep_hour") <= 10), 1)
    .when((col("dep_hour") >= 16) & (col("dep_hour") <= 20), 1)
    .otherwise(0)
)

df = df.withColumn(
    "is_weekend",
    when(col("DayOfWeek").isin([6, 7]), 1).otherwise(0)
)

df.select("dep_hour", "DayOfWeek", "is_peak_hour", "is_weekend").show(10)

+--------+---------+------------+----------+
|dep_hour|DayOfWeek|is_peak_hour|is_weekend|
+--------+---------+------------+----------+
|       8|        1|           1|         0|
|       8|        2|           1|         0|
|       8|        3|           1|         0|
|       8|        4|           1|         0|
|       8|        5|           1|         0|
|       8|        1|           1|         0|
|       8|        3|           1|         0|
|       8|        4|           1|         0|
|       8|        5|           1|         0|
|       8|        1|           1|         0|
+--------+---------+------------+----------+
only showing top 10 rows



## 4. Feature Selection

We avoid leakage columns such as actual arrival delay, actual departure delay, arrival time, departure time, and delay-cause columns because these values are not available before the flight happens.

Target variable:

- `is_arr_delayed`

In [59]:
model_df = df.select(
    "Month",
    "DayofMonth",
    "DayOfWeek",
    "Reporting_Airline",
    "Origin",
    "Dest",
    "OriginState",
    "DestState",
    "CRSDepTime",
    "CRSArrTime",
    "CRSElapsedTime",
    "Distance",
    "dep_hour",
    "season",
    "DepDelay",
    "DepDelayMinutes",
    "is_dep_delayed",
    "is_arr_delayed"
).dropna()

print("Modeling rows:", model_df.count())
print("Modeling columns:", len(model_df.columns))
model_df.show(5)

[Stage 1485:====>                                                 (1 + 10) / 11]

Modeling rows: 6952339
Modeling columns: 18
+-----+----------+---------+-----------------+------+----+-----------+---------+----------+----------+--------------+--------+--------+------+--------+---------------+--------------+--------------+
|Month|DayofMonth|DayOfWeek|Reporting_Airline|Origin|Dest|OriginState|DestState|CRSDepTime|CRSArrTime|CRSElapsedTime|Distance|dep_hour|season|DepDelay|DepDelayMinutes|is_dep_delayed|is_arr_delayed|
+-----+----------+---------+-----------------+------+----+-----------+---------+----------+----------+--------------+--------+--------+------+--------+---------------+--------------+--------------+
|    1|         8|        1|               9E|   LGA| OMA|         NY|       NE|       856|      1135|         219.0|  1148.0|       8|Winter|    -5.0|            0.0|             0|             0|
|    1|         9|        2|               9E|   LGA| OMA|         NY|       NE|       856|      1135|         219.0|  1148.0|       8|Winter|    -5.0|            0

## 6. Prepare Features for Machine Learning

Categorical columns are first indexed and then one-hot encoded.  
One-hot encoding is better for Logistic Regression because it avoids treating categories as ordered numbers.

In [60]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

categorical_cols = [
    "Reporting_Airline",
    "Origin",
    "Dest",
    "OriginState",
    "DestState",
    "season"
]

numeric_cols = [
    "Month",
    "DayofMonth",
    "DayOfWeek",
    "CRSDepTime",
    "CRSArrTime",
    "CRSElapsedTime",
    "Distance",
    "dep_hour",
    "DepDelay",
    "DepDelayMinutes",
    "is_dep_delayed"
]

label_col = "is_arr_delayed"

indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=c + "_idx",
        handleInvalid="keep"
    )
    for c in categorical_cols
]

encoder = OneHotEncoder(
    inputCols=[c + "_idx" for c in categorical_cols],
    outputCols=[c + "_vec" for c in categorical_cols]
)

feature_cols = [c + "_vec" for c in categorical_cols] + numeric_cols

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

In [61]:
from pyspark.sql.functions import col

# Split majority and minority classes
delayed_df = model_df.filter(col("is_arr_delayed") == 1)
not_delayed_df = model_df.filter(col("is_arr_delayed") == 0)

# Count delayed flights
delayed_count = delayed_df.count()
not_delayed_count = not_delayed_df.count()

print("Delayed:", delayed_count)
print("Not delayed:", not_delayed_count)

# Keep about 2x as many non-delayed flights as delayed flights
sample_fraction = min(1.0, (2 * delayed_count) / not_delayed_count)

not_delayed_sample = not_delayed_df.sample(
    withReplacement=False,
    fraction=sample_fraction,
    seed=42
)

balanced_df = delayed_df.union(not_delayed_sample)

print("Balanced rows:", balanced_df.count())
balanced_df.groupBy("is_arr_delayed").count().show()

Delayed: 1432402
Not delayed: 5519937


Balanced rows: 4298324


[Stage 1498:==========================>                          (11 + 10) / 22]

+--------------+-------+
|is_arr_delayed|  count|
+--------------+-------+
|             1|1432402|
|             0|2865922|
+--------------+-------+



## 7. Train-Test Split

In [62]:
train_df, test_df = balanced_df.randomSplit([0.8, 0.2], seed=42)

print("Training rows:", train_df.count())
print("Testing rows:", test_df.count())

Training rows: 3438964


[Stage 1504:==============================================>       (19 + 3) / 22]

Testing rows: 859360


## 8. Train Logistic Regression Model

Logistic Regression is used because it is scalable, simple, and provides probability estimates for arrival delay.

In [63]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.functions import vector_to_array
from pyspark.sql.functions import col, when

# Train Logistic Regression
lr = LogisticRegression(
    featuresCol="features",
    labelCol=label_col,
    maxIter=20,
    regParam=0.1,
    elasticNetParam=0.2
)

pipeline = Pipeline(stages=indexers + [encoder, assembler, lr])

lr_model = pipeline.fit(train_df)

# Get default predictions
predictions = lr_model.transform(test_df)

# Convert probability vector to array and extract probability of class 1 = delayed
predictions = predictions.withColumn(
    "prob_array",
    vector_to_array(col("probability"))
)

predictions = predictions.withColumn(
    "prob_delay",
    col("prob_array")[1]
)

# Lower threshold from 0.5 to 0.30 so model predicts delayed flights too
predictions = predictions.withColumn(
    "prediction",
    when(col("prob_delay") >= 0.4, 1.0).otherwise(0.0)
)

print("Model training completed.")

Model training completed.


In [68]:
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline

# For GBT, use StringIndexer only, not OneHotEncoder
gbt_feature_cols = [c + "_idx" for c in categorical_cols] + numeric_cols

gbt_assembler = VectorAssembler(
    inputCols=gbt_feature_cols,
    outputCol="features"
)

gbt = GBTClassifier(
    featuresCol="features",
    labelCol=label_col,
    maxIter=30,
    maxDepth=7,
    maxBins=400,
    stepSize=0.1
)

gbt_pipeline = Pipeline(stages=indexers + [gbt_assembler, gbt])

gbt_model = gbt_pipeline.fit(train_df)

gbt_predictions = gbt_model.transform(test_df)

print("GBT model trained.")

26/04/28 15:01:59 WARN DAGScheduler: Broadcasting large task binary with size 1014.6 KiB
26/04/28 15:01:59 WARN DAGScheduler: Broadcasting large task binary with size 1015.0 KiB
26/04/28 15:02:00 WARN DAGScheduler: Broadcasting large task binary with size 1015.6 KiB
26/04/28 15:02:00 WARN DAGScheduler: Broadcasting large task binary with size 1016.7 KiB
26/04/28 15:02:00 WARN DAGScheduler: Broadcasting large task binary with size 1022.6 KiB
26/04/28 15:02:00 WARN DAGScheduler: Broadcasting large task binary with size 1051.9 KiB
26/04/28 15:02:01 WARN DAGScheduler: Broadcasting large task binary with size 1114.9 KiB
26/04/28 15:02:01 WARN DAGScheduler: Broadcasting large task binary with size 1175.3 KiB
26/04/28 15:02:02 WARN DAGScheduler: Broadcasting large task binary with size 1175.8 KiB
26/04/28 15:02:02 WARN DAGScheduler: Broadcasting large task binary with size 1176.3 KiB
26/04/28 15:02:02 WARN DAGScheduler: Broadcasting large task binary with size 1177.5 KiB
26/04/28 15:02:02 WAR

GBT model trained.


In [69]:
from pyspark.ml.functions import vector_to_array
from pyspark.sql.functions import col, when

gbt_predictions = gbt_predictions.withColumn(
    "prob_array",
    vector_to_array(col("probability"))
)

gbt_predictions = gbt_predictions.withColumn(
    "prob_delay",
    col("prob_array")[1]
)

# Try threshold = 0.2 first
gbt_predictions = gbt_predictions.withColumn(
    "prediction",
    when(col("prob_delay") >= 0.2, 1.0).otherwise(0.0)
)

## 9. Evaluate Model

In [70]:
accuracy = MulticlassClassificationEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="accuracy"
).evaluate(gbt_predictions)

f1 = MulticlassClassificationEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="f1"
).evaluate(gbt_predictions)

auc = BinaryClassificationEvaluator(
    labelCol=label_col,
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
).evaluate(gbt_predictions)

print("Accuracy:", accuracy)
print("F1:", f1)
print("AUC:", auc)

26/04/28 15:03:56 WARN DAGScheduler: Broadcasting large task binary with size 4.4 MiB
26/04/28 15:04:04 WARN DAGScheduler: Broadcasting large task binary with size 4.4 MiB
26/04/28 15:04:12 WARN DAGScheduler: Broadcasting large task binary with size 4.4 MiB
                                                                                

Accuracy: 0.8730159653695774
F1: 0.8746318281315991
AUC: 0.9409367373732336


In [65]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

accuracy = MulticlassClassificationEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="accuracy"
).evaluate(predictions)

f1 = MulticlassClassificationEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="f1"
).evaluate(predictions)

precision = MulticlassClassificationEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="weightedPrecision"
).evaluate(predictions)

recall = MulticlassClassificationEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="weightedRecall"
).evaluate(predictions)

auc = BinaryClassificationEvaluator(
    labelCol=label_col,
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
).evaluate(predictions)

print("===== Logistic Regression Results =====")
print("Accuracy:", accuracy)
print("F1 Score:", f1)
print("Precision:", precision)
print("Recall:", recall)
print("AUC:", auc)

===== Logistic Regression Results =====
Accuracy: 0.9011508564513127
F1 Score: 0.899721415802837
Precision: 0.9006862344347671
Recall: 0.9011508564513125
AUC: 0.9208246981056192


## 10. Confusion Matrix

In [ ]:
conf_matrix = predictions.groupBy(label_col, "prediction").count().orderBy(label_col, "prediction")
conf_matrix.show()

In [71]:
conf_matrix = gbt_predictions.groupBy(label_col, "prediction").count().orderBy(label_col, "prediction")
conf_matrix.show()

26/04/28 15:04:51 WARN DAGScheduler: Broadcasting large task binary with size 4.5 MiB
26/04/28 15:04:58 WARN DAGScheduler: Broadcasting large task binary with size 4.4 MiB


+--------------+----------+------+
|is_arr_delayed|prediction| count|
+--------------+----------+------+
|             0|       0.0|501609|
|             0|       1.0| 71526|
|             1|       0.0| 37599|
|             1|       1.0|248626|
+--------------+----------+------+



26/04/28 17:33:09 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 1051793 ms exceeds timeout 120000 ms
26/04/28 17:33:09 WARN SparkContext: Killing executors is not supported by current scheduler.
26/04/28 17:48:23 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$

## 11. Sample Predictions

The `probability` column shows the estimated probability for each class:

- class 0: not delayed
- class 1: delayed

In [25]:
predictions.select(
    "Reporting_Airline",
    "Origin",
    "Dest",
    "Month",
    "DayOfWeek",
    "CRSDepTime",
    "Distance",
    "is_arr_delayed",
    "prediction",
    "probability"
).show(20, truncate=False)

[Stage 392:>                                                        (0 + 1) / 1]

+-----------------+------+----+-----+---------+----------+--------+--------------+----------+----------------------------------------+
|Reporting_Airline|Origin|Dest|Month|DayOfWeek|CRSDepTime|Distance|is_arr_delayed|prediction|probability                             |
+-----------------+------+----+-----+---------+----------+--------+--------------+----------+----------------------------------------+
|9E               |AEX   |ATL |1    |1        |600       |500.0   |0             |0.0       |[0.8412676292291522,0.15873237077084779]|
|9E               |ATL   |ABE |1    |1        |940       |692.0   |0             |0.0       |[0.8213625295206594,0.17863747047934064]|
|9E               |ATL   |AEX |1    |1        |1453      |500.0   |0             |0.0       |[0.7859625796133818,0.21403742038661822]|
|9E               |ATL   |AVL |1    |1        |1410      |164.0   |0             |0.0       |[0.7875701148767291,0.21242988512327088]|
|9E               |ATL   |CHO |1    |1        |1300    